In [392]:
import numpy as np
import pandas as pd
import random

In [ ]:
# --- 1. CONFIGURATION & HYPERPARAMETERS ---
ALPHA = 0.1    # Learning rate
GAMMA = 0.9    # Discount factor
EPSILON = 0.2  # Exploration rate
EPOCHS = 1000  # Iterations

# Total system capacity (Fixed pool)
TOTAL_CAPACITY = 1000 

# Actions: [0: No change, 1: Shift 50 RPS C->A, 2: Shift 50 RPS A->C]
ACTIONS = [0, 1, 2]

# States represent all possible ACTIONS the A and C can be
# States: (Tier_A_Load_Level, Tier_C_Waste_Level)
# Levels: 0 = Low (<30%), 1 = Mid (30-70%), 2 = High (>70%)
STATES = [(a, c) for a in range(3) for c in range(3)]
q_table = np.zeros((len(STATES), len(ACTIONS)))

print("STATES:",STATES)
print("q_table")
print(q_table)

# --- 2. HARDCODED MOCK DATA (Simulating K-Means/Stat-Inference) ---
# Each entry represents a 'snapshot' of the tiers provided by the other agents
environment_history = [
    {"a_usage": 480, "a_limit": 500, "c_usage": 20, "c_limit": 200}, # A is hungry, C is wasting
    {"a_usage": 100, "a_limit": 500, "c_usage": 180, "c_limit": 200}, # A is idle, C is heavy
    {"a_usage": 450, "a_limit": 500, "c_usage": 150, "c_limit": 200}, # Balanced
]

STATES: [(0, 0), (0, 1), (0, 2), (1, 0), (1, 1), (1, 2), (2, 0), (2, 1), (2, 2)]
q_table
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


In [ ]:
"""
State 2 (High Load): Usage is >70%. The agent sees this and thinks, Tier A is almost full, I might need to increase its quota.
State 1 (Mid Load): Usage is between 30% and 70%. This is the "Safe Zone.
State 0 (Low Load): Usage is <30%. Tier A is underutilized.
"""
def get_state_index(a_usage, a_limit, c_usage, c_limit):

    a_load = 2 if (a_usage/a_limit) > 0.7 else (1 if (a_usage/a_limit) > 0.3 else 0)
    
    c_waste = 2 if (c_usage/c_limit) < 0.3 else (1 if (c_usage/c_limit) < 0.7 else 0)
    
    print("   a_load  (> 0.7 = 2) | < 0.3 = 1):", a_usage/a_limit , a_load)
    print("   c_waste (< 0.3 = 2) | < 0.7 = 1):", c_usage/c_limit, c_waste)

    return STATES.index((a_load, c_waste))

In [394]:
# --- 3. REWARD FUNCTION ---
def calculate_reward(a_limit, a_usage, c_limit, c_usage):
    # Penalize dropped requests (starvation)
    starvation_penalty = max(0, a_usage - a_limit) * 2
    # Penalize unused quota (waste)
    waste_penalty = max(0, c_limit - c_usage) * 0.5
    # Reward throughput
    throughput_reward = min(a_usage, a_limit) + min(c_usage, c_limit)
    
    return (throughput_reward - starvation_penalty - waste_penalty) / 100

In [495]:
# --- 4. THE SIMULATION LOOP ---
print("Training Q-Learning Agent...")
EPOCHS = 1
for i in range(EPOCHS):
    # Pick a random scenario from our hardcoded history

    print("-----q_table-----")
    print(q_table)

    scenario = random.choice(environment_history)
    a_usage, c_usage = scenario['a_usage'], scenario['c_usage']
    current_a_limit, current_c_limit = scenario['a_limit'], scenario['c_limit']
    
    print("def get_state_index():")
    state_idx = get_state_index(a_usage, current_a_limit, c_usage, current_c_limit)
    print("state_idx:", state_idx)
    
    # Epsilon-greedy action selection
    if random.uniform(0, 1) < EPSILON:
        action = random.choice(ACTIONS)
        print("-----EXPLORATION-----")
    else:
        action = np.argmax(q_table[state_idx])
    
    print("action:", action)

    # Apply Action (Change limits)
    new_a_limit, new_c_limit = current_a_limit, current_c_limit
    if action == 1: # Shift C -> A
        new_a_limit += 50
        new_c_limit -= 50
    elif action == 2: # Shift A -> C
        new_a_limit -= 50
        new_c_limit += 50
        
    # Calculate Reward
    reward = calculate_reward(new_a_limit, a_usage, new_c_limit, c_usage)
    
    # Update Q-Table (Simplification: Next state is calculated based on current usage)
    print("def get_state_index(NEXT):")
    next_state_idx = get_state_index(a_usage, new_a_limit, c_usage, new_c_limit)
    print("next_state_idx:", next_state_idx)

    print("q_table[state_idx, action] :", state_idx, action ,q_table[state_idx, action] )
    q_table[state_idx, action] = q_table[state_idx, action] + ALPHA * (
        reward + GAMMA * np.max(q_table[next_state_idx]) - q_table[state_idx, action]
    )

print("Training Complete.\n")

Training Q-Learning Agent...
-----q_table-----
[[ 4.01064018  1.00614663  0.45515309]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [ 0.          0.          0.        ]
 [15.81363068  3.65217869  0.55      ]
 [ 0.          0.          0.        ]
 [10.97556184  3.1366472   0.368431  ]]
def get_state_index():
   a_load  (> 0.7 = 2) | < 0.3 = 1): 0.96 2
   c_waste (< 0.3 = 2) | < 0.7 = 1): 0.1 2
state_idx: 8
action: 0
def get_state_index(NEXT):
   a_load  (> 0.7 = 2) | < 0.3 = 1): 0.96 2
   c_waste (< 0.3 = 2) | < 0.7 = 1): 0.1 2
next_state_idx: 8
q_table[state_idx, action] : 8 0 10.975561844169695
Training Complete.



In [410]:
# --- 5. RESULTS: OPTIMAL POLICY ---
print("Learned Policy (State -> Best Action):")
print("State: (Tier A Load, Tier C Waste) | Action")
for i, state in enumerate(STATES):
    best_action = ACTIONS[np.argmax(q_table[i])]
    action_desc = "Do Nothing" if best_action == 0 else ("Shift C->A" if best_action == 1 else "Shift A->C")
    print(f"State {state}: {action_desc}")

Learned Policy (State -> Best Action):
State: (Tier A Load, Tier C Waste) | Action
State (0, 0): Do Nothing
State (0, 1): Do Nothing
State (0, 2): Do Nothing
State (1, 0): Do Nothing
State (1, 1): Do Nothing
State (1, 2): Do Nothing
State (2, 0): Do Nothing
State (2, 1): Do Nothing
State (2, 2): Do Nothing
